# Trabalho 2 - Agentes, MCP e Multiagente (Yelp Santa Barbara)
Rode as células **de cima para baixo**. Runtime: **GPU T4**. Cada bloco é idempotente: se a sessão cair, reabra e rode de novo (o avaliador e o juiz retomam de onde pararam).

## 1. Repositório e Drive

In [ ]:
REPO_URL = "https://github.com/SEU_USUARIO/SEU_REPO.git"   # <- ajuste
import os
if not os.path.exists("/content/repo"):
    !git clone $REPO_URL /content/repo
%cd /content/repo
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/yelp_recorte"   # pasta com business.csv, business_category.csv, users.csv, review.csv, friends.csv, reviewed.csv
os.environ["DRIVE_DIR"] = DRIVE_DIR
!ls $DRIVE_DIR

## 2. Instalar PostgreSQL, Neo4j, Ollama (qwen3:8b) e os dois ambientes Python (~10 min)

In [ ]:
!bash colab/01_preparar.sh

## 3. Carregar os dados nos dois bancos

In [ ]:
!python dados/carregar_bancos.py --dir "$DRIVE_DIR"

## 4. Subir o servidor MCP e validar SEM LLM (gera `resultados/validacao_mcp.json`)

In [ ]:
!bash colab/02_servidor.sh
!/content/venv_servidor/bin/python servidor/validar_servidor.py http://127.0.0.1:8765/mcp

## 5. Gerar as 20 perguntas com gabarito (revise o `avaliacao/perguntas.json` depois!)

In [ ]:
!python -m avaliacao.gerar_perguntas

## 6. Teste rápido (2 perguntas x 3 sistemas) antes de gastar tempo nas 60 execuções

In [ ]:
!python -m avaliacao.avaliador --limite 2 --saida resultados/smoke.csv
!rm -f resultados/smoke.csv

## 7. As 60 execuções (retoma se cair)

In [ ]:
!python -m avaliacao.avaliador

## 8. Juiz (Gemini). Guarde a chave em *Secrets* do Colab com o nome `GOOGLE_API_KEY`

In [ ]:
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
!python -m avaliacao.juiz

## 9. Análise (tabela, gráfico, erros por pergunta). Depois preencha `resultados/revisao_manual.csv` e rode de novo p/ a concordância

In [ ]:
!python -m avaliacao.analise
from IPython.display import Image
Image('resultados/grafico_acerto_custo.png')

## 10. Salvar tudo no Drive (a sessão do Colab é descartável)

In [ ]:
!mkdir -p /content/drive/MyDrive/yelp_resultados
!cp -r resultados avaliacao/perguntas.json /content/drive/MyDrive/yelp_resultados/